**Đọc file dataset**  https://www.kaggle.com/datasets/shivamb/go-emotions-google-emotions-dataset/data

In [1]:
!pip install -q kagglehub[pandas-datasets]

import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd

file_path = "go_emotions_dataset.csv"


df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "shivamb/go-emotions-google-emotions-dataset",
    file_path
)

# Khai báo cấu trúc hệ thống phân loại 28 nhãn chuẩn của Google
emotion_labels = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
    "joy", "love", "nervousness", "optimism", "pride", "realization",
    "relief", "remorse", "sadness", "surprise", "neutral"
]



/tmp/ipykernel_20770/2057014050.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'go-emotions-google-emotions-dataset' dataset.


**Gom cụm các rater** nếu > 2 người cùng đánh giá 1 câu cùng 1 loại cảm xúc thì giữ, còn lại bỏ

In [2]:
# Gom cụm các rater đánh giá cùng một câu bình luận lại với nhau
grouped = df.groupby(['id', 'text'])[emotion_labels].sum().reset_index()
print(f"Số lượng câu bình luận độc nhất sau khi gom cụm: {len(grouped)} câu.")

# Nhãn cảm xúc nào được ít nhất 2 chuyên gia chọn mới tính là 1, dưới 2 phiếu bầu sẽ bị hủy (= 0)
for col in emotion_labels:
    grouped[col] = (grouped[col] >= 2).astype(int)

# Tính tổng số nhãn được kích hoạt cho mỗi câu
grouped['has_label'] = grouped[emotion_labels].sum(axis=1)
# Chỉ giữ lại các câu có ít nhất 1 nhãn cảm xúc đạt sự đồng thuận từ 2 người trở lên
grouped_clean = grouped[grouped['has_label'] > 0].drop(columns=['has_label'])

print(f"🔹 Số lượng câu bình luận sạch thu được: {len(grouped_clean)} câu.")


Số lượng câu bình luận độc nhất sau khi gom cụm: 58011 câu.
🔹 Số lượng câu bình luận sạch thu được: 54263 câu.


**Làm sạch dữ liệu**

In [3]:
import nltk
from nltk.stem import WordNetLemmatizer
import re
import unicodedata

nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()

def advanced_clean_text(text):
    # 1. Tránh lỗi crash nếu dữ liệu bị trống (NaN)
    if not isinstance(text, str):
        return ""

    # 2. Đưa về chữ thường
    text = text.lower()

    # Bước A: Đồng bộ hóa mọi loại "dấu nháy"
    text = re.sub(r"[’‘`´]", "'", text)

    # Bước B: Xử lý chữ cái có dấu
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')

    # 3. Khai triển từ viết tắt
    text = re.sub(r"i'm", "i am", text)
    text = re.sub(r"don't", "do not", text)
    text = re.sub(r"doesn't", "does not", text)
    text = re.sub(r"can't", "cannot", text)
    text = re.sub(r"won't", "will not", text)
    text = re.sub(r"isn't", "is not", text)
    text = re.sub(r"aren't", "are not", text)
    text = re.sub(r"haven't", "have not", text)
    text = re.sub(r"hasn't", "has not", text)
    text = re.sub(r"didn't", "did not", text)
    text = re.sub(r"it's", "it is", text)
    text = re.sub(r"that's", "that is", text)
    text = re.sub(r"you're", "you are", text)
    text = re.sub(r"i've", "i have", text)
    text = re.sub(r"i'll", "i will", text)

    # Vớt những cụm viết tắt đuôi còn sót
    text = re.sub(r"n't", " not", text)
    text = re.sub(r"'re", " are", text)
    text = re.sub(r"'s", " is", text)
    text = re.sub(r"'d", " would", text)
    text = re.sub(r"'ll", " will", text)
    text = re.sub(r"'ve", " have", text)
    text = re.sub(r"'m", " am", text)

    # KỸ THUẬT NỐI TỪ PHỦ ĐỊNH
    text = re.sub(r"\bnot\s+([a-z]+)\b", r"not_\1", text)

    # 4. Lọc ký tự đặc biệt (giữ lại khoảng trắng, a-z, 0-9 và dấu biểu cảm ! ? _)
    text = re.sub(r"[^a-z0-9\s\!\?\_]", " ", text)

    # 5. Ép các ký tự lặp lại vô nghĩa (soooo -> so)
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # 6. Gom khoảng trắng thừa và cắt tỉa
    text = re.sub(r"\s+", " ", text).strip()

    # ========================================================
    # 7. KỸ THUẬT LEMMATIZATION (Ép gốc từ) - Đã cấy vào cuối hàm
    # ========================================================
    words = text.split()
    # pos='v' giúp ép các động từ (loved, loving, loves) về nguyên thể (love)
    lemmatized_words = [lemmatizer.lemmatize(w, pos='v') for w in words]
    text = " ".join(lemmatized_words)

    return text

# Chạy làm sạch trên tập dữ liệu
grouped_clean = grouped_clean.copy()
grouped_clean = grouped_clean.dropna(subset=['text'])
grouped_clean['text'] = grouped_clean['text'].apply(advanced_clean_text)
grouped_clean = grouped_clean[grouped_clean['text'] != ""]


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


**Chia tập train test valid**

In [4]:
from sklearn.model_selection import train_test_split

# 1. Tách ma trận đầu vào X (văn bản sạch) và ma trận đầu ra Y (28 nhãn cảm xúc)
X_text = grouped_clean['text'].astype(str).values
Y_matrix = grouped_clean[emotion_labels].values

# 2. Lần tách thứ nhất: Giữ lại 80% cho tập Train, đẩy 20% còn lại vào tập tạm (Temp)
X_train_text, X_temp_text, Y_train, Y_temp = train_test_split(
    X_text, Y_matrix, test_size=0.20, random_state=42
)

# 3. Lần tách thứ hai: Chia đôi 50/50 tập tạm (Temp) để lấy ra đúng 10% Dev và 10% Test biệt lập
X_val_text, X_test_text, Y_val, Y_test = train_test_split(
    X_temp_text, Y_temp, test_size=0.50, random_state=42
)

print(f"Tập Huấn luyện (Train - 80%): {X_train_text.shape[0]} câu.")
print(f"Tập Phát triển/Kiểm định (Val - 10%): {X_val_text.shape[0]} câu.")
print(f"Tập Kiểm thử (Test - 10%): {X_test_text.shape[0]} câu.")
print(f"Tổng cộng: {X_train_text.shape[0] + X_val_text.shape[0] + X_test_text.shape[0]} câu.")

Tập Huấn luyện (Train - 80%): 43408 câu.
Tập Phát triển/Kiểm định (Val - 10%): 5426 câu.
Tập Kiểm thử (Test - 10%): 5426 câu.
Tổng cộng: 54260 câu.


**TRÍCH XUẤT TF-IDF VECTORIZER**

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

print("SỐ HÓA VĂN BẢN BẰNG BỘ TRÍCH XUẤT TF-IDF VECTORIZER (BẢN NÂNG CẤP MAX/MIN DF)...")

# 1. Lấy danh sách stop_words mặc định và bảo vệ các từ phủ định
custom_stop_words = list(ENGLISH_STOP_WORDS)
words_to_keep = ['not', 'no', 'nor', 'none', 'cannot', 'nothing']

for w in words_to_keep:
    if w in custom_stop_words:
        custom_stop_words.remove(w)

# 2. Khởi tạo TF-IDF với BỘ LỌC ĐÁY và TRẦN
tfidf = TfidfVectorizer(
    max_features= 5000,
    stop_words=custom_stop_words,
    ngram_range=(1, 2),
    min_df=5,       # 🌟 LỌC ĐÁY: Bỏ qua các từ xuất hiện dưới 5 lần
    max_df=0.85     # 🌟 LỌC TRẦN: Bỏ qua các từ xuất hiện ở nhiều hơn 85% số câu
)

# 3. Tiến hành fit và transform trên tập Train
X_train_tfidf = tfidf.fit_transform(X_train_text)

# Chỉ transform (không fit lại) trên tập Val và Test
X_temp_tfidf = tfidf.transform(X_temp_text)
X_val_tfidf = tfidf.transform(X_val_text)
X_test_tfidf = tfidf.transform(X_test_text)

print(f"Kích thước ma trận TF-IDF tập Train: {X_train_tfidf.shape}")

SỐ HÓA VĂN BẢN BẰNG BỘ TRÍCH XUẤT TF-IDF VECTORIZER (BẢN NÂNG CẤP MAX/MIN DF)...
Kích thước ma trận TF-IDF tập Train: (43408, 5000)


**Mô hình Random Forest + TF-IDF**

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
import time

# =====================================================================
# BƯỚC 1: ĐỊNH NGHĨA MÔ HÌNH LÕI VỚI CÁC THAM SỐ CHỐNG OVERFITTING
# =====================================================================
base_rf_model = RandomForestClassifier(
    class_weight='balanced',
    n_estimators=100,         # Trồng 100 cây quyết định
    max_depth=50,             # Hạ độ sâu xuống 50 để tránh học vẹt tập Train
    min_samples_split=10,     # Cành chỉ chẻ tiếp nếu có ít nhất 10 câu văn
    min_samples_leaf=2,       # Ngọn lá cuối cùng phải đọng lại ít nhất 2 câu văn
    n_jobs=1,                 # Giữ n_jobs=1 để nhường CPU cho bộ bọc OvR bên ngoài
    random_state=42
)

# =====================================================================
# BƯỚC 2: BỌC MÔ HÌNH BẰNG CHIẾN LƯỢC ONE-VS-REST (OvR)
# =====================================================================
# Kích hoạt n_jobs=-1 ở đây để dùng toàn bộ nhân CPU huấn luyện 28 nhãn cùng lúc
ovr_classifier = OneVsRestClassifier(base_rf_model, n_jobs=-1)

start_time = time.time()

print("Đang tiến hành huấn luyện hệ thống 28 Rừng Ngẫu Nhiên (OvR)... (Vui lòng kiên nhẫn nhé!)")

# BƯỚC 3: TIẾN HÀNH HUẤN LUYỆN
ovr_classifier.fit(X_train_tfidf, Y_train)

end_time = time.time()

print("-" * 70)
print(f"✅ Thời gian mô hình học xong toàn bộ dữ liệu: {end_time - start_time:.2f} giây.")
print("✅ Trạng thái: Hệ thống phân loại 28 nhãn OvR đã sẵn sàng ra quyết định!")

Đang tiến hành huấn luyện hệ thống 28 Rừng Ngẫu Nhiên (OvR)... (Vui lòng kiên nhẫn nhé!)
----------------------------------------------------------------------
✅ Thời gian mô hình học xong toàn bộ dữ liệu: 96.06 giây.
✅ Trạng thái: Hệ thống phân loại 28 nhãn OvR đã sẵn sàng ra quyết định!


**Đánh giá trên tập test**

In [7]:
from sklearn.metrics import f1_score, classification_report
import numpy as np

print("🤖 ĐANG CHẠY THUẬT TOÁN TÌM NGƯỠNG TỐI ƯU TỰ ĐỘNG TRÊN TẬP VAL...")
print("-" * 70)

# =====================================================================
# BƯỚC 1: LẤY XÁC SUẤT THÔ TRÊN TẬP VAL (ĐỂ TÌM NGƯỠNG)
# =====================================================================
# Sử dụng đúng tên biến mô hình từ bước huấn luyện trước
Y_val_proba_list = ovr_classifier.predict_proba(X_val_tfidf)

# Trích xuất cột xác suất của class 1 (Tỉ lệ % thuộc về nhãn đó)
# Y_val_proba_list is already a 2D array where each element (i,j) is P(class 1 | sample i, label j).
# So, we can directly assign it.
Y_val_proba = Y_val_proba_list

# Từ điển lưu ngưỡng tốt nhất của từng nhãn
best_thresholds = {}

# Vòng lặp quét qua từng nhãn trong 28 nhãn
for idx, label in enumerate(emotion_labels):
    best_f1 = 0.0
    best_thresh = 0.5  # Ngưỡng an toàn mặc định

    # Quét thử các ngưỡng từ 0.1 đến 0.95 (mỗi bước nhảy 0.05)
    # Dùng np.round để tránh lỗi trôi dạt số thập phân của float
    for thresh in np.round(np.arange(0.1, 1.0, 0.05), 2):
        # Dự đoán thử với ngưỡng này trên tập Val
        preds = (Y_val_proba[:, idx] >= thresh).astype(int)

        # Tính thử điểm F1-score của riêng nhãn này
        score = f1_score(Y_val[:, idx], preds, zero_division=0)

        # Cập nhật nếu F1 cao hơn
        if score > best_f1:
            best_f1 = score
            best_thresh = thresh

    best_thresholds[label] = best_thresh

print("✅ Đã tìm xong bộ ngưỡng tối ưu cho 28 nhãn dựa trên toán học!")
print("📋 Một số ngưỡng tối ưu tìm được:")
# In gọn 5 nhãn đầu để kiểm tra, tránh làm rối màn hình console
for k in emotion_labels[:5]:
    print(f"   🔹 Sắc thái '{k}': Ngưỡng tối ưu = {best_thresholds[k]:.2f}")
print("   🔹 ... (và 23 nhãn khác)")
print("-" * 70)

🤖 ĐANG CHẠY THUẬT TOÁN TÌM NGƯỠNG TỐI ƯU TỰ ĐỘNG TRÊN TẬP VAL...
----------------------------------------------------------------------
✅ Đã tìm xong bộ ngưỡng tối ưu cho 28 nhãn dựa trên toán học!
📋 Một số ngưỡng tối ưu tìm được:
   🔹 Sắc thái 'admiration': Ngưỡng tối ưu = 0.55
   🔹 Sắc thái 'amusement': Ngưỡng tối ưu = 0.55
   🔹 Sắc thái 'anger': Ngưỡng tối ưu = 0.60
   🔹 Sắc thái 'annoyance': Ngưỡng tối ưu = 0.50
   🔹 Sắc thái 'approval': Ngưỡng tối ưu = 0.55
   🔹 ... (và 23 nhãn khác)
----------------------------------------------------------------------


### 🚀 TRIỂN KHAI GIAO DIỆN DEMO VỚI GRADIO

In [8]:
!pip install -q gradio

In [9]:
import gradio as gr
import numpy as np

def predict_emotion(text):
    if not text.strip():
        return "Vui lòng nhập văn bản!"

    # 1. Làm sạch văn bản đầu vào
    cleaned_text = advanced_clean_text(text)

    # 2. Chuyển đổi sang vector TF-IDF
    text_tfidf = tfidf.transform([cleaned_text])

    # 3. Lấy xác suất từ mô hình Random Forest (OvR)
    # predict_proba trả về mảng 2D (n_samples, n_classes)
    probas = ovr_classifier.predict_proba(text_tfidf)[0]

    # 4. Trình bày kết quả dựa trên bộ ngưỡng tối ưu
    results = {}
    activated = False

    for idx, prob in enumerate(probas):
        label = emotion_labels[idx]
        # Lấy ngưỡng tương ứng từ dictionary best_thresholds
        threshold_val = best_thresholds[label]

        if prob >= threshold_val:
            results[label] = float(prob)
            activated = True

    # Nếu không nhãn nào vượt ngưỡng, lấy nhãn có xác suất cao nhất để đảm bảo luôn có kết quả
    if not activated:
        top_idx = np.argmax(probas)
        results[emotion_labels[top_idx]] = float(probas[top_idx])

    return results

# Khởi tạo giao diện Gradio
iface = gr.Interface(
    fn=predict_emotion,
    inputs=gr.Textbox(lines=2, placeholder="Nhập câu tiếng Anh tại đây...", label="Văn bản đầu vào"),
    outputs=gr.Label(num_top_classes=5, label="Phân loại cảm xúc (Top 5)"),
    title="🚀 Google GoEmotions Predictor (Random Forest)",
    description="Nhận diện 28 sắc thái cảm xúc dựa trên Random Forest và bộ ngưỡng tối ưu hóa trên tập Validation.",
    examples=[
        ["I am so excited!"],
        ["This is so disappointing and annoying."],
        ["Thank you so much for your help, I really appreciate it!"],
        ["I feel a bit nervous about the upcoming exam."],
        ["That is a very interesting perspective, I am curious to learn more."]
    ]
)

# Khởi chạy giao diện
iface.launch(share=True, inline=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1bccb115ce38721be5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
